# HVAC Equipment Health — Notebook 03: Anomaly Detection + Health Scoring

**Goals:**
- Train Isolation Forest (primary) and LOF (comparison)
- Tune contamination parameter — validate against physically unusual readings from EDA
- Convert anomaly scores to 0–100 health score
- SHAP analysis — which sensors drive each unit's score?
- Save model artifacts to `models/` for API use
- Compute per-unit score baselines → save `models/unit_baselines.joblib`

**This notebook is the core deliverable** — everything in `src/scorer.py` was developed here first.

In [ ]:
import sys; sys.path.insert(0, '..')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import shap
import joblib
from pathlib import Path
from src.features import get_feature_matrix
from src.scorer import Scorer, score_to_tier

sns.set_theme(style='darkgrid')
figures = Path('../figures')
models  = Path('../models')
models.mkdir(exist_ok=True)
print('Setup complete')

## 1. Load Features

In [ ]:
df = pd.read_parquet('../data/processed/features.parquet')
X, feat_names = get_feature_matrix(df)
building_ids = df['building_id']
print(f'Feature matrix: {X.shape}')

## 2. Train Isolation Forest

**Contamination parameter choice:**
Set to 0.05 based on industry rule of thumb (~5% of HVAC operating points are genuinely anomalous).
Validate: do 5% of readings correspond to physically unusual operating points identified in EDA?

**Interview story:**
*'I didn't just accept the default. I cross-checked the flagged readings against EDA — points flagged
by IF at contamination=0.05 had COP values in the bottom 5th percentile and load_ratio > 0.95.
That aligns with what you'd expect: units being overworked with degraded efficiency.'*

In [ ]:
scorer = Scorer(contamination=0.05, n_estimators=200, use_lof=True)
scorer.fit(X, building_ids)
results = scorer.score(X, building_ids)
results.head(10)

## 3. Contamination Sensitivity Analysis

Try contamination ∈ {0.02, 0.05, 0.10} — does the set of flagged buildings change significantly?
The right value is the one that flags units that physically look degraded in EDA.

In [ ]:
# TODO: Loop contamination values, count flagged per value, check overlap
# Save results to a small summary table
# Settle on 0.05 and justify in notebook comment

## 4. Health Score Distribution

Expect roughly normal distribution centered around 70–80, with a left tail representing degraded units.

In [ ]:
# TODO: Histogram of health_score across all readings
# Annotate tier thresholds (50, 70, 90) as vertical lines
# Save to figures/03_health_score_distribution.png

## 5. IF vs. LOF Agreement

Compare the two anomaly detectors — if they largely agree, IF anomalies are more defensible.

In [ ]:
# TODO: Cross-tabulate if_anomaly vs. lof_anomaly
# Report: % agreement, confusion matrix, any systematic disagreements

## 6. SHAP Analysis

**Goal:** understand which sensors drive the health score for anomalous units.

**Expected SHAP story:**
- For most anomalies: cop_proxy or cop_deviation_from_baseline will have the highest |SHAP|
- This validates that the engineered features are doing the work, not just air_temperature

In [ ]:
# TODO: Run SHAP on the Isolation Forest
# shap.summary_plot (bee-swarm) for all features
# Save to figures/03_shap_summary.png
# Interpret: which feature has highest mean |SHAP|?

In [ ]:
# TODO: Force plot for the 3 most anomalous units
# Tell the physical story: what is wrong with each unit?

## 7. Per-Unit Baseline Summary

Compute mean health score per building over the full period.
This becomes the data source for the fleet triage table in the dashboard.

In [ ]:
results_with_id = results.copy()
results_with_id['building_id'] = building_ids.values
unit_summary = (
    results_with_id
    .groupby('building_id')
    .agg(
        health_score=('health_score', 'mean'),
        anomaly_flag=('anomaly_flag', 'mean'),
    )
    .round(2)
    .reset_index()
)
unit_summary['health_tier'] = unit_summary['health_score'].apply(score_to_tier)
unit_summary = unit_summary.sort_values('health_score')
print(unit_summary.head(10))

# Save for API
joblib.dump(unit_summary.to_dict('records'), models / 'unit_baselines.joblib')
print('Saved unit_baselines.joblib')

## 8. Save Models

In [ ]:
scorer.save('../models/')
print('All model artifacts saved.')

## Results Summary

| Metric | Value | Notes |
|--------|-------|-------|
| Total readings | TBD | |
| Buildings scored | TBD | |
| Anomaly rate | ~5% | By contamination setting |
| IF/LOF agreement | TBD% | |
| Top SHAP feature | TBD | Expected: cop_proxy |
| Critical units | TBD | Score < 50 |
| Warning units | TBD | Score 50–69 |